# امیج جنریشن ایپلیکیشنز بنانا (OpenAI)

LLMs صرف متن کی تخلیق تک محدود نہیں ہیں۔ آپ متن کی وضاحتوں سے تصاویر بھی بنا سکتے ہیں۔ تصاویر بطور ایک شکل میڈٹیک، فنِ تعمیر، سیاحت، گیم ڈویلپمنٹ، مارکیٹنگ، اور دیگر شعبوں میں مفید ہیں۔ اس سبق میں ہم OpenAI پلیٹ فارم پر آج کے **GPT Image** ماڈلز کے ساتھ کام کرتے ہیں۔

## سیکھنے کے مقاصد

اس سبق کے اختتام تک آپ قابل ہو جائیں گے:

- وضاحت کریں کہ امیج جنریشن کیا ہے اور کہاں مفید ہے۔
- `gpt-image` ماڈل خاندان کو سمجھیں اور اس کی وراثتی DALL·E ماڈلز سے کیا فرق ہے۔
- ایک امیج جنریشن ایپلیکیشن بنائیں اور تصاویر میں ترمیم کریں۔

## امیج جنریشن کیا ہے؟

امیج جنریشن ماڈلز متن کی تحریر سے تصاویر بناتے ہیں۔ جدید ماڈلز جیسے کہ `gpt-image` تربیت کے دوران متن اور تصاویر کے تعلق کو سیکھتے ہیں، پھر تدریجی انداز میں بے ترتیب شور کو ایسی تصویر میں تبدیل کرتے ہیں جو آپ کے پرامپٹ سے میل کھاتی ہو۔

امیج ماڈلز کے دو مشہور خاندان ہیں:

- **`gpt-image` (OpenAI)** — موجودہ نسل جو اس سبق میں استعمال ہو رہی ہے۔ یہ ٹیکسٹ سے امیج بنانے اور تصاویر میں ترمیم (ماسک کے ساتھ ان پینٹنگ) کی حمایت کرتا ہے۔
- **Midjourney** — ایک مقبول تھرڈ پارٹی ماڈل ہے جس کی اپنی سروس اور Discord پر مبنی ورک فلو ہے۔

> پرانے OpenAI امیج ماڈلز — **DALL·E 2** اور **DALL·E 3** — وراثتی ہیں۔ DALL·E 3 نئے تعینات کرنے کے لیے دستیاب نہیں ہے، اور `create_variation` جیسی خصوصیات صرف DALL·E 2 میں موجود تھیں۔ نئے ایپلیکیشنز کے لیے `gpt-image` ماڈلز استعمال کریں۔

> **اہم:** `gpt-image` ماڈلز تیار شدہ تصویر کو **base64** (`b64_json`) کے طور پر واپس کرتے ہیں، URL کے طور پر نہیں۔ آپ کا کوڈ base64 سٹرنگ کو بائٹس میں ڈیکوڈ کرتا ہے اور محفوظ کرتا ہے — یہاں کوئی تصویر URL ڈاؤن لوڈ کرنے کے لیے نہیں ہے۔


## اپنی پہلی امیج جنریشن ایپلیکیشن بنانا

تو ایک امیج جنریشن ایپلیکیشن بنانے کے لیے کیا چاہیے؟ آپ کو درج ذیل لائبریریاں درکار ہیں:

- **python-dotenv**، آپ کو تجویز کی جاتی ہے کہ اپنے راز *.env* فائل میں کوڈ سے دور رکھیں اس لائبریری کو استعمال کریں۔
- **openai**، یہ لائبریری OpenAI API کے ساتھ تعامل کرنے کے لیے استعمال کریں گے۔
- **pillow**، پائتھن میں تصاویر کے ساتھ کام کرنے کے لیے۔


1. ایک *.env* فائل بنائیں جس میں درج ذیل مواد ہو:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. اوپر دی گئی لائبریریز کو *requirements.txt* نامی فائل میں اس طرح جمع کریں:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. اس کے بعد، ورچوئل ماحول بنائیں اور لائبریریز انسٹال کریں:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> ونڈوز کے لیے، اپنا ورچوئل ماحول بنانے اور فعال کرنے کے لیے درج ذیل کمانڈز استعمال کریں:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* نامی فائل میں درج ذیل کوڈ شامل کریں:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # OpenAI آبجیکٹ بنائیں (اپنے .env سے OPENAI_API_KEY پڑھتا ہے)
    client = openai.OpenAI()


    try:
        # امیج جنریشن API کا استعمال کرتے ہوئے تصویر بنائیں
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # یہاں اپنی پرامپٹ ٹیکسٹ درج کریں
            size='1024x1024',
            n=1
        )
        # محفوظ شدہ تصویر کے لیے ڈائریکٹری سیٹ کریں
        image_dir = os.path.join(os.curdir, 'images')

        # اگر ڈائریکٹری موجود نہیں ہے تو اسے بنائیں
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # تصویر کا راستہ شروع کریں (نوٹ کریں کہ فائل کی قسم png ہونی چاہیے)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image ماڈلز تصویر کو base64 (b64_json) میں واپس کرتے ہیں، URL نہیں
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # تصویر کو ڈیفالٹ امیج ویور میں دکھائیں
        image = Image.open(image_path)
        image.show()

    # استثنیات کو پکڑیں
    except openai.BadRequestError as err:
        print(err)

    ```

آئیے اس کوڈ کی وضاحت کرتے ہیں:

- سب سے پہلے، ہم وہ لائبریریاں امپورٹ کرتے ہیں جن کی ہمیں ضرورت ہے، جن میں OpenAI لائبریری، dotenv لائبریری، base64 ماڈیول، اور Pillow لائبریری شامل ہیں۔

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- اس کے بعد، ہم کلائنٹ بناتے ہیں، جو آپ کی ``.env`` سے API کلید پڑھتا ہے۔

    ```python
    # اوپن اے آئی آبجیکٹ بنائیں
    client = openai.OpenAI()
    ```

- اگلا، ہم تصویر بناتے ہیں:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # یہاں اپنا پرامپٹ متن درج کریں
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` ماڈلز تصویر کو **base64** اسٹرنگ کے طور پر `data[0].b64_json` میں لوٹاتے ہیں۔ ہم اسے بائٹس میں ڈی کوڈ کرتے ہیں اور فائل میں لکھ دیتے ہیں — ڈاؤن لوڈ کرنے کے لیے کوئی URL نہیں ہے۔

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- آخر میں، ہم تصویر کھولتے ہیں اور اسے دکھانے کے لیے معیاری امیج ویوئر استعمال کرتے ہیں:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### تصویر تیار کرنے کی مزید تفصیلات

آئیے `images.generate` کے پیرامیٹرز دیکھتے ہیں:

- **model**، استعمال ہونے والا تصویر ماڈل ہے (مثال کے طور پر `gpt-image-1`)۔
- **prompt**، وہ متنی پرامپٹ ہے جس سے تصویر بنتی ہے۔ یہاں "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" ہے۔
- **size**، پیدا کی گئی تصویر کا سائز ہے (مثال کے طور پر `1024x1024`، `1536x1024`، `1024x1536`، یا `"auto"`)۔
- **n**، پیدا کردہ تصاویر کی تعداد ہے۔ یہاں ہم ایک تصویر پیدا کرتے ہیں۔

> تصویر کے ماڈلز `temperature` پیرا میٹر نہیں لیتے — یہ ٹیکسٹ جنریشن کنٹرول ہے۔ تغیر حاصل کرنے کے لیے API کو دوبارہ کال کریں؛ تغیر کم کرنے کے لیے، اپنا پرامپٹ زیادہ مخصوص بنائیں۔

## تصویر بنانے کی اضافی صلاحیتیں

آپ نے دیکھا کہ چند لائنز پائتھون کی مدد سے تصویر کیسے بنائی جاتی ہے۔ `gpt-image` ماڈلز موجودہ تصویر میں **ترمیم** بھی کر سکتے ہیں۔ ایک تصویر، ایک اختیاری **ماسک** (جو تبدیل کرنے والے علاقے کی نشاندہی کرتا ہے)، اور ایک پرامپٹ فراہم کرکے، آپ تصویر کے کسی حصے کو بدل سکتے ہیں — مثال کے طور پر، خرگوش کے لیے ٹوپی شامل کریں۔

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# ترمیمات بھی بیس64 کے طور پر واپس کی جاتی ہیں
edited_image = base64.b64decode(response.data[0].b64_json)
```

بنیادی تصویر میں صرف خرگوش ہے؛ آخری تصویر میں خرگوش کے سر پر ٹوپی ہے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
